In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

In [ ]:
# =========================
# CONFIG (edit these)
# =========================

DE_DIR = r"/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/07_perturb_seq_T_cells"
PERT_H5AD = r"/ShangGaoAIProjects/Zang/single_cell_data/scRNA_reference_dataset/T_cell_perturb_seq/GWCD4i.DE_stats.h5ad"

OUTDIR = os.path.join(DE_DIR, "step2_driver_ranking")
os.makedirs(OUTDIR, exist_ok=True)

# --------
# Your DE CSVs (make sure these exist)
# --------
DE_2TO3     = os.path.join(DE_DIR, "DE_cluster2_vs_cluster3.fullV2.csv")   # computed as 3 vs 2
DE_4TO3     = os.path.join(DE_DIR, "DE_cluster4_vs_cluster3.fullV2.csv")   # computed as 3 vs 4
DE_2TO4     = os.path.join(DE_DIR, "DE_cluster2_vs_cluster4.fullV2.csv")   # computed as 4 vs 2
DE_C3_RVSNR = os.path.join(DE_DIR, "DE_cluster3_R_vs_NR.fullV2.csv")       # within C3: R vs NR

# If you only have NR_vs_R, set these and turn on FLIP_SIGNATURE_SIGN below.
# DE_4_RVSNR = os.path.join(DE_DIR, "DE_cluster4_NR_vs_R.full.csv")
# DE_1_RVSNR = os.path.join(DE_DIR, "DE_cluster1_NR_vs_R.full.csv")

# If your file is NR_vs_R but you want NR->R, you should flip sign (multiply signature by -1)
FLIP_SIGNATURE_SIGN_FOR = set()
# --------
# Signature thresholds
# --------
SIG_FDR = 0.05
SIG_ABS_LFC = 0.25
SIG_TOPK = 500

# --------
# Perturb-seq selection
# --------
CONDITION_KEEP = None  # None or one of {"Rest","Stim8hr","Stim48hr"}
EFFECT_LAYER = "zscore"  # or "log_fc"

# --------
# QC filters
# --------
USE_QC_FILTERS = True
QC_MIN_CELLS_TARGET = 50
QC_REQUIRE_ONTARGET_SIGNIFICANT = True
QC_EXCLUDE_OFFTARGET_FLAG = True

# --------
# Optional gene filtering
# --------
FILTER_DOWNSTREAM_BY_BASEMEAN = True
DOWNSTREAM_BASEMEAN_MIN = 0.1

# --------
# Aggregation
# --------
AGG_FUNC = "mean"  # "mean" or "median"

# --------
# Perturb-seq obs columns (from your inspection)
# --------
PERT_COL = "target_contrast_gene_name"
COND_COL = "culture_condition"

print("OUTDIR:", OUTDIR)


In [ ]:
def load_signature_from_de(csv_path, name, fdr=0.05, abs_lfc=0.25, topk=500, flip_sign=False):
    """
    Build a signature vector s in gene-symbol space from scanpy DE csv:
      S_g = sign(logFC) * |logFC|   for genes passing thresholds, else 0.
    """
    df = pd.read_csv(csv_path)
    required = {"names", "logfoldchanges"}
    if not required.issubset(df.columns):
        raise ValueError(f"{csv_path} missing required columns {required}. Found: {df.columns.tolist()[:20]}")

    df = df.copy()
    df["gene"] = df["names"].astype(str)

    # filter
    if "pvals_adj" in df.columns:
        df = df[df["pvals_adj"] <= fdr]
    df = df[df["logfoldchanges"].abs() >= abs_lfc]

    # prioritize by abs LFC
    df["abs_lfc"] = df["logfoldchanges"].abs()
    df = df.sort_values(["abs_lfc"], ascending=False).head(topk)

    sign = np.sign(df["logfoldchanges"].values).astype(float)
    weight = df["abs_lfc"].values.astype(float)

    sig = pd.DataFrame({"gene": df["gene"].tolist(), "sign": sign, "weight": weight})
    sig["S"] = sig["sign"] * sig["weight"]
    if flip_sign:
        sig["S"] = -sig["S"]
    sig = sig.drop_duplicates("gene").set_index("gene")
    sig.name = name
    return sig


def cosine_scores(X, s_vec):
    """cosine similarity between each row of X and vector s_vec."""
    s = s_vec.astype(np.float64)
    s_norm = np.sqrt(np.sum(s * s) + 1e-12)

    if sparse.issparse(X):
        X = X.tocsr()
        num = X.dot(s)
        X_sq = X.multiply(X).sum(axis=1)
        X_norm = np.sqrt(np.asarray(X_sq).reshape(-1) + 1e-12)
        return np.asarray(num).reshape(-1) / (X_norm * s_norm + 1e-12)

    X = np.asarray(X, dtype=np.float64)
    num = X @ s
    X_norm = np.sqrt(np.sum(X * X, axis=1) + 1e-12)
    return num / (X_norm * s_norm + 1e-12)


def build_ensg_by_symbol(p):
    """Map gene symbol -> one Ensembl ID in p.var_names, using p.var['gene_name']."""
    if "gene_name" not in p.var.columns:
        raise ValueError("p.var missing 'gene_name' column.")
    sym = p.var["gene_name"].astype(str).values
    ensg = p.var_names.astype(str).values
    ensg_by_symbol = pd.Series(ensg, index=sym).drop_duplicates(keep="first")
    ensg_by_symbol = ensg_by_symbol[ensg_by_symbol.index.notna()]
    ensg_by_symbol = ensg_by_symbol[ensg_by_symbol.index != "nan"]
    ensg_by_symbol = ensg_by_symbol[ensg_by_symbol.index != ""]
    return ensg_by_symbol


def apply_qc_filters(p):
    """Filter perturb-seq rows by QC columns in p.obs."""
    if not USE_QC_FILTERS:
        return p

    keep = np.ones(p.n_obs, dtype=bool)

    if "n_cells_target" in p.obs.columns and QC_MIN_CELLS_TARGET is not None:
        keep &= (p.obs["n_cells_target"].astype(float).values >= float(QC_MIN_CELLS_TARGET))

    if QC_REQUIRE_ONTARGET_SIGNIFICANT and "ontarget_significant" in p.obs.columns:
        keep &= (p.obs["ontarget_significant"].astype(bool).values)

    if QC_EXCLUDE_OFFTARGET_FLAG and "offtarget_flag" in p.obs.columns:
        keep &= (~p.obs["offtarget_flag"].astype(bool).values)

    return p[keep].copy()


def filter_genes_by_basemean(p_sub):
    """
    Optionally filter genes by baseMean.
    For speed, use mean across rows for sparse; median for dense.
    """
    if not FILTER_DOWNSTREAM_BY_BASEMEAN:
        return p_sub
    if "baseMean" not in p_sub.layers:
        return p_sub

    bm = p_sub.layers["baseMean"]
    if sparse.issparse(bm):
        bm = bm.tocsr()
        col_stat = np.asarray(bm.mean(axis=0)).reshape(-1)
    else:
        col_stat = np.median(np.asarray(bm), axis=0)

    keep_cols = col_stat >= float(DOWNSTREAM_BASEMEAN_MIN)
    if keep_cols.sum() < 50:
        return p_sub
    return p_sub[:, keep_cols].copy()


def score_signature(p, sig, ensg_by_symbol, effect_layer):
    """
    Align signature (symbol) with perturb data (ENSG columns), then cosine score per row.
    Returns: scores (n_obs,), n_genes_used
    """
    common_symbols = ensg_by_symbol.index.intersection(sig.index)
    if len(common_symbols) < 30:
        raise ValueError(f"Too few overlap genes for signature '{sig.name}': {len(common_symbols)}")

    common_ensg = ensg_by_symbol.loc[common_symbols].values
    p_sub = p[:, common_ensg].copy()
    p_sub = filter_genes_by_basemean(p_sub)

    # match signature to p_sub column order via gene_name
    sym_sub = p_sub.var["gene_name"].astype(str).values
    S = sig.loc[sym_sub, "S"].values.astype(np.float64)

    X = p_sub.layers[effect_layer]
    scores = cosine_scores(X, S)
    return scores, len(sym_sub)


def aggregate_scores(res, group_cols, agg_func="mean"):
    agg = "median" if agg_func == "median" else "mean"
    return (
        res.groupby(group_cols)
           .agg(
               n=("score_2to3", "size"),
               score_2to3=("score_2to3", agg),
               score_4to3=("score_4to3", agg),
               score_2to4=("score_2to4", agg),
               score_C3_NR_to_R=("score_C3_NR_to_R", agg),
           )
           .reset_index()
    )

def save_top(df, score_col, out_name, topn=50):
    d = df.sort_values(score_col, ascending=False).head(topn).copy()
    d.to_csv(os.path.join(OUTDIR, out_name), index=False)
    return d


In [ ]:
# New 4 contrasts (directions already oriented as destination vs source)
sig_2to3 = load_signature_from_de(
    DE_2TO3, "2_to_3",   # computed as 3 vs 2
    fdr=SIG_FDR, abs_lfc=SIG_ABS_LFC, topk=SIG_TOPK,
    flip_sign=False
)

sig_4to3 = load_signature_from_de(
    DE_4TO3, "4_to_3",   # computed as 3 vs 4
    fdr=SIG_FDR, abs_lfc=SIG_ABS_LFC, topk=SIG_TOPK,
    flip_sign=False
)

sig_2to4 = load_signature_from_de(
    DE_2TO4, "2_to_4",   # computed as 4 vs 2 (exhaustion-branch direction)
    fdr=SIG_FDR, abs_lfc=SIG_ABS_LFC, topk=SIG_TOPK,
    flip_sign=False
)

sig_c3_nr2r = load_signature_from_de(
    DE_C3_RVSNR, "C3_NR_to_R",  # computed as R vs NR within cluster3
    fdr=SIG_FDR, abs_lfc=SIG_ABS_LFC, topk=SIG_TOPK,
    flip_sign=False
)

print("Signature sizes:", {
    "2to3": sig_2to3.shape[0],
    "4to3": sig_4to3.shape[0],
    "2to4": sig_2to4.shape[0],
    "C3_NR_to_R": sig_c3_nr2r.shape[0],
})


In [ ]:
sig_2to3

In [ ]:
p = sc.read_h5ad(PERT_H5AD)
print("Perturb AnnData:", p)
print("Pert obs columns:", p.obs.columns.tolist())
print("Pert var columns:", p.var.columns.tolist())
print("Pert layers:", list(p.layers.keys()))

# sanity check columns
for col in [PERT_COL, COND_COL]:
    if col not in p.obs.columns:
        raise ValueError(f"Missing p.obs['{col}']. Available: {p.obs.columns.tolist()}")

if EFFECT_LAYER not in p.layers:
    raise ValueError(f"EFFECT_LAYER='{EFFECT_LAYER}' not found. Available layers: {list(p.layers.keys())}")

# optional condition keep
if CONDITION_KEEP is not None:
    p = p[p.obs[COND_COL].astype(str) == str(CONDITION_KEEP)].copy()
    print("Filtered to condition:", CONDITION_KEEP, "n_obs =", p.n_obs)

# QC filtering
p_before = p.n_obs
p = apply_qc_filters(p)
print(f"QC filters applied: n_obs {p_before} -> {p.n_obs}")

ensg_by_symbol = build_ensg_by_symbol(p)
print("Symbol->ENSG map size:", len(ensg_by_symbol))


In [ ]:
scores_2to3, n_2to3 = score_signature(p, sig_2to3, ensg_by_symbol, EFFECT_LAYER)
scores_4to3, n_4to3 = score_signature(p, sig_4to3, ensg_by_symbol, EFFECT_LAYER)
scores_2to4, n_2to4 = score_signature(p, sig_2to4, ensg_by_symbol, EFFECT_LAYER)
scores_c3,   n_c3   = score_signature(p, sig_c3_nr2r, ensg_by_symbol, EFFECT_LAYER)

res = p.obs[[PERT_COL, COND_COL]].copy()
res["score_2to3"] = scores_2to3
res["score_4to3"] = scores_4to3
res["score_2to4"] = scores_2to4
res["score_C3_NR_to_R"] = scores_c3



In [ ]:
focus = ["EOMES","PRDM1","STAT1","FOXO3","BATF","AHR","RELB","STAT5A",
         "ARNTL","NR1D2","NFKB1","NFATC1","NFATC2","IRF8","RUNX1","TBX21"]


In [ ]:
res.loc[res['target_contrast_gene_name'].isin(focus), :]

In [ ]:
genes_all = p.var["gene_name"].astype(str).tolist()   # 用 symbol 空间！


In [ ]:
LAYER = "zscore"   # 或 "log_fc"
Delta = p.layers[LAYER]   # shape: (n_obs, n_vars)

In [ ]:
import numpy as np
import pandas as pd

def get_stat1_top_genes_from_perturb(p, cond="Stim8hr", top_n_each=25, fdr_max=0.05,
                                    layer_lfc="log_fc", layer_fdr="adj_p_value"):
    m = (p.obs["target_contrast_gene_name"].astype(str).values == "SREBF2") & \
        (p.obs["culture_condition"].astype(str).values == cond)
    idx = np.where(m)[0]
    if len(idx) == 0:
        raise ValueError(f"No RORA row found in p.obs for condition={cond}")
    i = idx[0]

    genes = p.var["gene_name"].astype(str).values
    lfc  = np.asarray(p.layers[layer_lfc][i]).ravel()
    fdr  = np.asarray(p.layers[layer_fdr][i]).ravel()

    df = pd.DataFrame({"gene": genes, "log_fc": lfc, "adj_p": fdr})
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    df = df[df["adj_p"] <= fdr_max].copy()
    if df.empty:
        raise ValueError(f"No genes pass FDR<={fdr_max} for RORA@{cond}. Try fdr_max=0.1")

    up = df[df["log_fc"] > 0].sort_values("log_fc", ascending=False).head(top_n_each)
    dn = df[df["log_fc"] < 0].sort_values("log_fc", ascending=True).head(top_n_each)

    return dn["gene"].tolist(), up["gene"].tolist(), df

genes_dn, genes_up, df_stat1_all = get_stat1_top_genes_from_perturb(
    p, cond="Stim8hr", top_n_each=20, fdr_max=0.05
)
genes_stat1 = genes_dn + genes_up
print("ARNTL downstream genes used:", len(genes_stat1))
print("Top down:", genes_dn[:20])
print("Top up:", genes_up[:20])


In [ ]:
import numpy as np
import pandas as pd

def l2_normalize(v):
    v = np.asarray(v, float)
    return v / (np.linalg.norm(v) + 1e-12)

def cosine(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    return float(np.dot(a, b) / ((np.linalg.norm(a) + 1e-12) * (np.linalg.norm(b) + 1e-12)))

def build_sig_from_sigdf(sig_df, genes_all, score_col="S", top_n=250):
    """
    sig_df: index=gene, 有一列 score_col (比如 'S')
    返回：
      v_sig (len=genes_all, L2 norm),
      sig_genes (list),
      sig_weights (np.array, 已按top_n筛过)
    """
    genes_all = list(map(str, genes_all))
    gset = set(genes_all)

    d = sig_df.copy()
    d = d.reset_index().rename(columns={"index":"gene"})
    d["gene"] = d["gene"].astype(str)
    d[score_col] = d[score_col].astype(float)

    # 只保留在 perturb-seq gene space 的基因
    d = d[d["gene"].isin(gset)].copy()

    # 选 top_n（按 |S|）
    d = d.sort_values(score_col, key=lambda x: x.abs(), ascending=False).head(top_n)

    sig_genes = d["gene"].tolist()
    sig_weights = d[score_col].to_numpy(dtype=float)

    g2i = {g:i for i,g in enumerate(genes_all)}
    v = np.zeros(len(genes_all), float)
    for g, w in zip(sig_genes, sig_weights):
        v[g2i[g]] = w
    v = l2_normalize(v)
    return v, sig_genes, sig_weights

In [ ]:
v23, g23, w23 = build_sig_from_sigdf(sig_2to3, genes_all, score_col="S", top_n=250)
v24, g24, w24 = build_sig_from_sigdf(sig_2to4, genes_all, score_col="S", top_n=250)
vRR, gRR, wRR = build_sig_from_sigdf(sig_c3_nr2r, genes_all, score_col="S", top_n=250)

In [ ]:
def perm_null_cos(delta, genes_all, sig_weights, n_genes, n_perm=800, seed=0):
    """
    随机抽 n_genes 个基因，把 sig_weights（固定分布）贴上去，算 cos(delta, v_rand)
    注意：sig_weights 的长度必须等于 n_genes
    """
    rng = np.random.default_rng(seed)
    genes_all = np.array(list(map(str, genes_all)), dtype=object)
    g2i = {g:i for i,g in enumerate(genes_all)}

    w = np.asarray(sig_weights, float)
    w = w / (np.linalg.norm(w) + 1e-12)

    null = np.zeros(n_perm, float)
    for k in range(n_perm):
        pick = rng.choice(genes_all, size=n_genes, replace=False)
        v = np.zeros(len(genes_all), float)
        for gg, ww in zip(pick, w):
            v[g2i[gg]] = ww
        null[k] = cosine(delta, v)
    return null

def empirical_p_and_z(obs, null, two_sided=True):
    null = np.asarray(null, float)
    if two_sided:
        p = (1 + np.sum(np.abs(null) >= abs(obs))) / (len(null) + 1)
    else:
        p = (1 + np.sum(null >= obs)) / (len(null) + 1)
    z = (obs - null.mean()) / (null.std(ddof=0) + 1e-12)
    lo, hi = np.quantile(null, [0.025, 0.975])
    return float(p), float(z), float(lo), float(hi)


In [ ]:
import numpy as np
import pandas as pd

def row_delta(p, tf, cond, layer="zscore"):
    m = (p.obs["target_contrast_gene_name"].astype(str).values == tf) & \
        (p.obs["culture_condition"].astype(str).values == cond)
    idx = np.where(m)[0]
    if len(idx) == 0:
        return None
    # 理论上应该只有一行；如果有多行就取第一行或取均值
    i = idx[0]
    d = np.asarray(p.layers[layer][i]).ravel()
    return d

def validate_focus_TFs_in_DE_adata(
    p,
    focus_tfs,
    conds=("Rest","Stim8hr","Stim48hr"),
    layer="zscore",
    v23=None, v24=None, vRR=None,
    w23=None, w24=None, wRR=None,
    n_perm=800,
    seed=0
):
    genes_all = p.var["gene_name"].astype(str).tolist()

    rows = []
    for tf in focus_tfs:
        for cond in conds:
            delta = row_delta(p, tf, cond, layer=layer)
            row = {
                "TF": tf, "culture_condition": cond, "status":"OK",
                "push_obs": np.nan, "push_p": np.nan, "push_z": np.nan, "push_null_lo": np.nan, "push_null_hi": np.nan,
                "rescue_obs": np.nan, "rescue_p": np.nan, "rescue_z": np.nan, "rescue_null_lo": np.nan, "rescue_null_hi": np.nan
            }
            if delta is None or not np.isfinite(delta).any():
                row["status"] = "MISSING"
                rows.append(row)
                continue

            # observed
            s23 = cosine(delta, v23)
            s24 = cosine(delta, v24)
            push_obs = s23 - s24
            rescue_obs = cosine(delta, vRR)

            # null (gene-set matched random signatures)
            null23 = perm_null_cos(delta, genes_all, w23, n_genes=len(w23), n_perm=n_perm, seed=seed+11)
            null24 = perm_null_cos(delta, genes_all, w24, n_genes=len(w24), n_perm=n_perm, seed=seed+23)
            push_null = null23 - null24
            nullR  = perm_null_cos(delta, genes_all, wRR, n_genes=len(wRR), n_perm=n_perm, seed=seed+37)

            p_push, z_push, lo_push, hi_push = empirical_p_and_z(push_obs, push_null, two_sided=True)
            p_res,  z_res,  lo_res,  hi_res  = empirical_p_and_z(rescue_obs, nullR,  two_sided=True)

            row.update({
                "push_obs": push_obs, "push_p": p_push, "push_z": z_push, "push_null_lo": lo_push, "push_null_hi": hi_push,
                "rescue_obs": rescue_obs, "rescue_p": p_res, "rescue_z": z_res, "rescue_null_lo": lo_res, "rescue_null_hi": hi_res
            })
            rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
focus_tfs = ["FOXO3","RELB","BATF","STAT1","NR1D2","TBX21"]

df_val = validate_focus_TFs_in_DE_adata(
    p=p,
    focus_tfs=focus_tfs,
    conds=("Rest","Stim8hr","Stim48hr"),
    layer="zscore",        # 推荐
    v23=v23, v24=v24, vRR=vRR,
    w23=w23, w24=w24, wRR=wRR,
    n_perm=800,
    seed=0
)

df_val.to_csv("/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/figures/perturbseq_focusTF_randomBG.csv", index=False)

df_val[df_val["status"]=="OK"].sort_values(["push_p","rescue_p"]).head(20)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _pstars(p):
    if not np.isfinite(p): return ""
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

def plot_STAT1_from_df_val(df_val,
                           out_png,
                           cond_order=("Rest","Stim8hr","Stim48hr"),
                           title="Perturb-seq supports STAT1 as a fate-bias driver (condition-specific)",
                           show_numeric_p=True):
    d = df_val.copy()
    d = d[(d["status"]=="OK") & (d["TF"].astype(str)=="STAT1")].copy()
    d["culture_condition"] = pd.Categorical(d["culture_condition"].astype(str),
                                            categories=list(cond_order), ordered=True)
    d = d.sort_values("culture_condition")
    if d.empty:
        raise ValueError("No STAT1 rows found in df_val (status==OK).")

    x = np.arange(len(cond_order), dtype=float)

    fig, axes = plt.subplots(2, 1, figsize=(7.2, 6.2), sharex=True)

    def panel(ax, y_col, lo_col, hi_col, p_col, ylabel, panel_title):
        y  = d[y_col].astype(float).to_numpy()
        lo = d[lo_col].astype(float).to_numpy()
        hi = d[hi_col].astype(float).to_numpy()
        pp = d[p_col].astype(float).to_numpy()

        # 灰色 null 区间：竖条 [lo,hi]
        for xi, l, h in zip(x, lo, hi):
            ax.plot([xi, xi], [l, h], linewidth=10, alpha=0.25, solid_capstyle="butt", color="grey")

        # observed 点+线
        ax.plot(x, y, marker="o", linewidth=2.8, color="black", label="Observed (STAT1)")

        # 0 线
        ax.axhline(0, linewidth=1, color="black", alpha=0.7)

        # 标注 p
        for xi, yi, pi in zip(x, y, pp):
            tag = _pstars(pi)
            if show_numeric_p and np.isfinite(pi):
                txt = f"{tag}\np={pi:.3g}"
            else:
                txt = tag
            ax.text(xi, yi, txt, ha="center", va="bottom", fontsize=10)

        ax.set_ylabel(ylabel)
        ax.set_title(panel_title)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    panel(
        axes[0],
        y_col="push_obs", lo_col="push_null_lo", hi_col="push_null_hi", p_col="push_p",
        ylabel="Observed score\n(vs null 95% interval)",
        panel_title="A  Fate-bias alignment (push)"
    )

    panel(
        axes[1],
        y_col="rescue_obs", lo_col="rescue_null_lo", hi_col="rescue_null_hi", p_col="rescue_p",
        ylabel="Observed score\n(vs null 95% interval)",
        panel_title="B  Rescue alignment (NR→R)  [negative control]"
    )

    axes[1].set_xticks(x)
    axes[1].set_xticklabels(cond_order)
    axes[1].set_xlabel("Culture condition (Perturb-seq)")

    fig.suptitle(title, y=1.02, fontsize=13)
    plt.tight_layout()

    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("[DONE] saved:", out_png)


# 用法
out_png = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/figures/Fig_STAT1_PerturbSeq_pvalue_support.png"
plot_STAT1_from_df_val(df_val, out_png)


In [ ]:
import numpy as np
import pandas as pd

genes_query = ["TOX", "PDCD1", "HAVCR2"]          # PD-1=PDCD1; TIM-3=HAVCR2
conds = ["Rest", "Stim8hr", "Stim48hr"]
layer_fc = "log_fc"                               # 或者 "zscore"
layer_z  = "zscore"

# gene name -> var index
gene_name = p.var["gene_name"].astype(str).values
g2i = {g:i for i,g in enumerate(gene_name)}

missing = [g for g in genes_query if g not in g2i]
print("[Missing genes in p.var['gene_name']]:", missing)

rows = []
for cond in conds:
    m = (p.obs["target_contrast_gene_name"].astype(str).values == "STAT1") & \
        (p.obs["culture_condition"].astype(str).values == cond)
    idx = np.where(m)[0]
    if len(idx) == 0:
        print(f"[WARN] No STAT1 row for condition {cond}")
        continue
    i = idx[0]  # 通常只有一行
    for g in genes_query:
        if g not in g2i:
            continue
        j = g2i[g]
        rows.append({
            "TF": "STAT1",
            "culture_condition": cond,
            "gene": g,
            "log_fc": float(p.layers[layer_fc][i, j]),
            "zscore": float(p.layers[layer_z][i, j]),
        })

df_stat1 = pd.DataFrame(rows)
df_stat1 = df_stat1.sort_values(["gene", "culture_condition"])
print(df_stat1)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings

# ----------------------------
# Basic matplotlib config
# ----------------------------
mpl.rcParams["font.family"] = "Liberation Sans"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
for qc_col in ["n_cells_target", "ontarget_significant", "offtarget_flag"]:
    if qc_col in p.obs.columns:
        res[qc_col] = p.obs[qc_col].values

res.to_csv(os.path.join(OUTDIR, "perturb_scores_per_obs.csv"), index=False)
print("saved:", os.path.join(OUTDIR, "perturb_scores_per_obs.csv"))


In [ ]:
agg_tc = aggregate_scores(res, [PERT_COL, COND_COL], agg_func=AGG_FUNC)
agg_tc.to_csv(os.path.join(OUTDIR, "driver_scores_agg_by_target_condition.csv"), index=False)

agg_t = aggregate_scores(res, [PERT_COL], agg_func=AGG_FUNC)
agg_t.to_csv(os.path.join(OUTDIR, "driver_scores_agg_by_target.csv"), index=False)

print("saved agg tables to OUTDIR:", OUTDIR)

# =========================
# 5) add a therapy-oriented meta score
#     (promote 2->3, 4->3, C3 NR->R; penalize 2->4)
# =========================
agg_t["meta_score"] = (agg_t["score_2to3"] + agg_t["score_4to3"] + agg_t["score_C3_NR_to_R"] - agg_t["score_2to4"]) / 4.0
agg_t = agg_t.sort_values("meta_score", ascending=False)

agg_t.to_csv(os.path.join(OUTDIR, "driver_scores_agg_by_target_with_meta.csv"), index=False)
agg_t.head(50).to_csv(os.path.join(OUTDIR, "top_meta_drivers.csv"), index=False)
print("saved meta ranking:", os.path.join(OUTDIR, "top_meta_drivers.csv"))

In [ ]:
import pandas as pd

path = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/07_perturb_seq_T_cells/step2_driver_ranking/driver_scores_agg_by_target_with_meta.csv"
df = pd.read_csv(path)

# 可选：先过滤掉 n 太小的 target（更稳）
df = df[df["n"] >= 3].copy()

topk = 30

top_2to3 = df.sort_values("score_2to3", ascending=False).head(topk)
top_4to3 = df.sort_values("score_4to3", ascending=False).head(topk)
top_c3   = df.sort_values("score_C3_NR_to_R", ascending=False).head(topk)
top_2to4 = df.sort_values("score_2to4", ascending=False).head(topk)  # 这是“促耗竭”榜单
top_meta = df.sort_values("meta_score", ascending=False).head(topk)

top_2to3.to_csv("top_score_2to3.csv", index=False)
top_4to3.to_csv("top_score_4to3.csv", index=False)
top_c3.to_csv("top_score_C3_NR_to_R.csv", index=False)
top_2to4.to_csv("top_score_2to4_pro_exhaustion.csv", index=False)
top_meta.to_csv("top_meta_therapy_candidates.csv", index=False)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# INPUT: your top lists
# =========================
TOP_META = r"/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/07_perturb_seq_T_cells/top_meta_therapy_candidates.csv"
TOP_2TO3 = r"/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/07_perturb_seq_T_cells/top_score_2to3.csv"
TOP_4TO3 = r"/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/07_perturb_seq_T_cells/top_score_4to3.csv"
TOP_2TO4 = r"/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/07_perturb_seq_T_cells/top_score_2to4_pro_exhaustion.csv"
TOP_C3   = r"/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/07_perturb_seq_T_cells/top_score_C3_NR_to_R.csv"

OUTDIR = r"/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/07_perturb_seq_T_cells/stepABC_outputs"
os.makedirs(OUTDIR, exist_ok=True)

# =========================
# Common column names (must exist in your csvs)
# =========================
GENE_COL = "target_contrast_gene_name"
SCORE_COLS = ["score_2to3", "score_4to3", "score_2to4", "score_C3_NR_to_R", "meta_score"]

# -------------------------
# Load
# -------------------------
df_meta = pd.read_csv(TOP_META)
df_2to3 = pd.read_csv(TOP_2TO3)
df_4to3 = pd.read_csv(TOP_4TO3)
df_2to4 = pd.read_csv(TOP_2TO4)
df_c3   = pd.read_csv(TOP_C3)

# sanity check
for name, df in [("meta", df_meta), ("2to3", df_2to3), ("4to3", df_4to3), ("2to4", df_2to4), ("C3", df_c3)]:
    missing = [c for c in [GENE_COL] + SCORE_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"[{name}] missing columns: {missing}. Found: {df.columns.tolist()}")

# =========================
# STEP A: overlap / consensus analysis
# =========================
set_meta = set(df_meta[GENE_COL].astype(str))
set_2to3 = set(df_2to3[GENE_COL].astype(str))
set_4to3 = set(df_4to3[GENE_COL].astype(str))
set_2to4 = set(df_2to4[GENE_COL].astype(str))
set_c3   = set(df_c3[GENE_COL].astype(str))

# (A1) key intersections
consensus_push_rescue = sorted(set_meta & set_2to3 & set_4to3)  # “推效应 + 救回耗竭 + meta”
consensus_push_rescue_c3 = sorted(set_meta & set_2to3 & set_4to3 & set_c3)
pro_exhaustion_overlap = sorted(set_2to4 & set_meta)  # meta 与促耗竭榜单的交集（理论上应很少）

# save lists
pd.DataFrame({GENE_COL: consensus_push_rescue}).to_csv(os.path.join(OUTDIR, "A_consensus_meta_2to3_4to3.csv"), index=False)
pd.DataFrame({GENE_COL: consensus_push_rescue_c3}).to_csv(os.path.join(OUTDIR, "A_consensus_meta_2to3_4to3_C3.csv"), index=False)
pd.DataFrame({GENE_COL: pro_exhaustion_overlap}).to_csv(os.path.join(OUTDIR, "A_overlap_meta_and_pro_exhaustion_2to4.csv"), index=False)

# (A2) membership table for all genes appearing in any top list
all_genes = sorted(set_meta | set_2to3 | set_4to3 | set_2to4 | set_c3)
mem = pd.DataFrame({GENE_COL: all_genes})
mem["in_meta"] = mem[GENE_COL].isin(set_meta)
mem["in_2to3"] = mem[GENE_COL].isin(set_2to3)
mem["in_4to3"] = mem[GENE_COL].isin(set_4to3)
mem["in_2to4_pro_exhaustion"] = mem[GENE_COL].isin(set_2to4)
mem["in_C3_NR_to_R"] = mem[GENE_COL].isin(set_c3)
mem["n_lists"] = mem[[ "in_meta","in_2to3","in_4to3","in_2to4_pro_exhaustion","in_C3_NR_to_R"]].sum(axis=1)

mem = mem.sort_values(["n_lists", GENE_COL], ascending=[False, True])
mem.to_csv(os.path.join(OUTDIR, "A_membership_table_all_genes.csv"), index=False)

# (A3) overlap summary table
summary = pd.DataFrame([
    {"name": "meta", "n": len(set_meta)},
    {"name": "2to3", "n": len(set_2to3)},
    {"name": "4to3", "n": len(set_4to3)},
    {"name": "2to4_pro_exhaustion", "n": len(set_2to4)},
    {"name": "C3_NR_to_R", "n": len(set_c3)},
    {"name": "meta∩2to3∩4to3", "n": len(consensus_push_rescue)},
    {"name": "meta∩2to3∩4to3∩C3", "n": len(consensus_push_rescue_c3)},
    {"name": "meta∩2to4_pro_exhaustion", "n": len(pro_exhaustion_overlap)},
])
summary.to_csv(os.path.join(OUTDIR, "A_overlap_summary.csv"), index=False)

print("[Step A] saved to:", OUTDIR)
print(summary)



In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from adjustText import adjust_text

# =========================
# CONFIG: set your paths
# =========================
OUTDIR = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/07_perturb_seq_T_cells"
os.makedirs(OUTDIR, exist_ok=True)

# Step A outputs (pass QC top drivers per objective)
TOP_2TO3 = os.path.join(OUTDIR, "top_score_2to3.csv")
TOP_4TO3 = os.path.join(OUTDIR, "top_score_4to3.csv")
TOP_2TO4 = os.path.join(OUTDIR, "top_score_2to4_pro_exhaustion.csv")
TOP_C3   = os.path.join(OUTDIR, "top_score_C3_NR_to_R.csv")

# Optional: consensus list from Step A (if you want to label them)
CONSENSUS4 = ["CD320", "FBXW7", "PARP8", "RANBP6"]

DPI = 300
GENE_COL = "target_contrast_gene_name"

# score columns expected in each CSV
COL_2TO3 = "score_2to3"
COL_4TO3 = "score_4to3"
COL_2TO4 = "score_2to4"
COL_C3   = "score_C3_NR_to_R"

# =========================
# helpers
# =========================
def _read_top(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    if GENE_COL not in df.columns:
        raise ValueError(f"{path} missing column '{GENE_COL}'. Columns={df.columns.tolist()}")
    df[GENE_COL] = df[GENE_COL].astype(str)
    return df

def _ensure_cols(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df

def _union_score_table(df2, df4, df24, dfc3) -> pd.DataFrame:
    """
    Build a gene x 4-score table by taking the best available row per gene from each top list.
    """
    # Keep only (gene + relevant score col) then merge
    d2  = df2[[GENE_COL, COL_2TO3]].drop_duplicates(GENE_COL)
    d4  = df4[[GENE_COL, COL_4TO3]].drop_duplicates(GENE_COL)
    d24 = df24[[GENE_COL, COL_2TO4]].drop_duplicates(GENE_COL)
    dc3 = dfc3[[GENE_COL, COL_C3]].drop_duplicates(GENE_COL)

    M = d2.merge(d4, on=GENE_COL, how="outer") \
          .merge(d24, on=GENE_COL, how="outer") \
          .merge(dc3, on=GENE_COL, how="outer")

    # Fill missing scores with 0 for heatmap visualization (means "not selected / not in top list")
    for c in [COL_2TO3, COL_4TO3, COL_2TO4, COL_C3]:
        if c not in M.columns:
            M[c] = 0.0
    M[[COL_2TO3, COL_4TO3, COL_2TO4, COL_C3]] = M[[COL_2TO3, COL_4TO3, COL_2TO4, COL_C3]].fillna(0.0)
    return M

def _sort_genes_for_heatmap(M: pd.DataFrame) -> pd.DataFrame:
    """
    Sort by (effector restoration signal) and then by overall magnitude.
    We want: high 2to3 + high 4to3 + low 2to4 (pro-exhaustion) to appear near top.
    """
    m = M.copy()
    m["restoration_index"] = m[COL_2TO3] + m[COL_4TO3] - m[COL_2TO4]
    m["magnitude"] = np.sqrt((m[[COL_2TO3, COL_4TO3, COL_2TO4, COL_C3]] ** 2).sum(axis=1))
    m = m.sort_values(["restoration_index", "magnitude"], ascending=[False, False])
    return m.drop(columns=["restoration_index", "magnitude"])

# =========================
# FIGURE 1: heatmap
# =========================
def plot_heatmap_all_top_passQC(
    top_2to3_path: str,
    top_4to3_path: str,
    top_2to4_path: str,
    top_c3_path: str,
    out_png: str,
    *,
    max_genes: int | None = None,   # set e.g. 120 if too tall
):
    df2  = _read_top(top_2to3_path)
    df4  = _read_top(top_4to3_path)
    df24 = _read_top(top_2to4_path)
    dfc3 = _read_top(top_c3_path)

    M = _union_score_table(df2, df4, df24, dfc3)
    M = _sort_genes_for_heatmap(M)

    if max_genes is not None and len(M) > max_genes:
        M = M.head(max_genes).copy()

    genes = M[GENE_COL].tolist()
    X = M[[COL_2TO3, COL_4TO3, COL_2TO4, COL_C3]].to_numpy(dtype=float)

    # symmetric color scale around 0
    vmax = float(np.nanmax(np.abs(X))) if X.size else 1.0
    vmax = max(vmax, 1e-6)

    # dynamic figure height
    fig_h = max(4.5, 0.22 * len(genes))
    fig_w = 7.8

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(X, aspect="auto", interpolation="nearest", vmin=-vmax, vmax=vmax)

    ax.set_title("Top perturb-seq drivers across four objectives (pass QC)", loc="left", fontweight="bold")
    ax.set_xticks(range(4))
    ax.set_xticklabels(
        [
            "score_2to3\n(C3 vs C2)",
            "score_4to3\n(C3 vs C4)",
            "score_2to4\n(C4 vs C2)",
            "score_C3_NR_to_R\n(within C3)",
        ],
        rotation=0,
    )

    ax.set_yticks(range(len(genes)))
    ax.set_yticklabels(genes, fontsize=8)

    # light gridlines between cells (optional but usually looks cleaner)
    ax.set_xticks(np.arange(-.5, 4, 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(genes), 1), minor=True)
    ax.grid(which="minor", linewidth=0.4)
    ax.tick_params(which="minor", bottom=False, left=False)

    cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label("Perturbation–signature cosine score", rotation=90)

    plt.tight_layout()
    plt.savefig(out_png, dpi=DPI)
    plt.close(fig)

# =========================
# FIGURE 2: Panel B scatter
# =========================
def plot_panelB_push_vs_rescue(
    top_2to3_path: str,
    top_4to3_path: str,
    top_2to4_path: str,
    out_png: str,
    *,
    label_genes: list[str] | None = None,
    label_topk_pro: int = 6,
    consensus4: list[str] | None = None,
):
    df2  = _read_top(top_2to3_path)
    df4  = _read_top(top_4to3_path)
    df24 = _read_top(top_2to4_path)

    # Build a "prioritized_drivers" set: union of top push (2to3) and top rescue (4to3)
    pri_genes = set(df2[GENE_COL].astype(str)) | set(df4[GENE_COL].astype(str))
    pro_genes = set(df24[GENE_COL].astype(str))

    # Build one table with both x/y scores available
    # Some genes are only in df2 or df4 -> fill missing with 0 (or NaN; here 0 makes plotting stable)
    M = _union_score_table(df2, df4, df24, dfc3=_ensure_cols(df2.copy(), [COL_C3]))  # dummy c3
    # Remove dummy col
    if COL_C3 in M.columns:
        M = M.drop(columns=[COL_C3])

    M["group"] = "other"
    M.loc[M[GENE_COL].isin(pri_genes), "group"] = "prioritized_drivers"
    M.loc[M[GENE_COL].isin(pro_genes), "group"] = "pro_exhaustion_drivers"

    pri = M[M["group"] == "prioritized_drivers"].copy()
    pro = M[M["group"] == "pro_exhaustion_drivers"].copy()

    fig, ax = plt.subplots(figsize=(10, 7))

    ax.scatter(pri[COL_2TO3], pri[COL_4TO3], label="prioritized_drivers", s=60)
    ax.scatter(pro[COL_2TO3], pro[COL_4TO3], label="pro_exhaustion_drivers", s=60)

    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)

    ax.set_title("Effector-push vs effector-rescue scores")
    ax.set_xlabel("score_2to3 (transitional→effector)")
    ax.set_ylabel("score_4to3 (exhausted→effector)")
    ax.legend(frameon=True)

    # choose genes to label
    if consensus4 is None:
        consensus4 = []
    consensus4 = [g for g in consensus4 if g in set(M[GENE_COL])]

    if label_genes is None:
        # default: label consensus + top pro-exhaustion by |2to4|
        top_pro = df24.sort_values(COL_2TO4, ascending=False).head(label_topk_pro)[GENE_COL].astype(str).tolist()
        label_genes = []
        label_genes += consensus4
        label_genes += [g for g in top_pro if g not in label_genes]
    else:
        # keep only those present
        label_genes = [g for g in label_genes if g in set(M[GENE_COL])]

    texts = []
    for g in label_genes:
        r = M[M[GENE_COL] == g].iloc[0]
        x = float(r[COL_2TO3])
        y = float(r[COL_4TO3])
        texts.append(
            ax.text(
                x, y, g,
                fontsize=11,
                fontweight="bold" if g in set(consensus4) else None
            )
        )

    # repel so labels don't overlap
    if texts:
        adjust_text(
            texts,
            ax=ax,
            arrowprops=dict(arrowstyle="-", linewidth=0.8),
            expand_text=(1.12, 1.25),
            expand_points=(1.12, 1.25),
            force_text=(0.2, 0.7),
            force_points=(0.1, 0.4),
            lim=600,
        )

    plt.tight_layout()
    plt.savefig(out_png, dpi=DPI)
    plt.close(fig)

# =========================
# RUN
# =========================
if __name__ == "__main__":
    out_heat = os.path.join(OUTDIR, "fig_heatmap_top_drivers_passQC.png")
    out_b    = os.path.join(OUTDIR, "panelB_push_vs_rescue_no_overlap.png")

    plot_heatmap_all_top_passQC(
        TOP_2TO3, TOP_4TO3, TOP_2TO4, TOP_C3,
        out_png=out_heat,
        max_genes=None,   # set e.g. 120 if it gets too tall
    )
    print("[saved]", out_heat)

    plot_panelB_push_vs_rescue(
        TOP_2TO3, TOP_4TO3, TOP_2TO4,
        out_png=out_b,
        consensus4=CONSENSUS4,
        # 你也可以手动指定 label_genes 更可控：
        # label_genes=["CD320","FBXW7","PARP8","RANBP6","FOXO1","UBE2L3","ZNF106","PRKAR1A","SIK3","SGF29"],
        label_genes=None,
        label_topk_pro=6,
    )
    print("[saved]", out_b)


In [ ]:
GENE_COL = "target_contrast_gene_name"
COLS = ["score_2to3", "score_4to3", "score_2to4", "score_C3_NR_to_R", "meta_score"]

df_meta = pd.read_csv(TOP_META)
df_2to3 = pd.read_csv(TOP_2TO3)
df_4to3 = pd.read_csv(TOP_4TO3)
df_2to4 = pd.read_csv(TOP_2TO4)
df_c3   = pd.read_csv(TOP_C3)

# ====== build a unified table for union genes (since each list is top30 only) ======
def idx(df):
    tmp = df[[GENE_COL] + COLS].copy()
    tmp[GENE_COL] = tmp[GENE_COL].astype(str)
    return tmp.drop_duplicates(GENE_COL).set_index(GENE_COL)

idx_meta = idx(df_meta)
idx_2to3 = idx(df_2to3)
idx_4to3 = idx(df_4to3)
idx_2to4 = idx(df_2to4)
idx_c3   = idx(df_c3)

all_genes = sorted(set(df_meta[GENE_COL].astype(str)) |
                   set(df_2to3[GENE_COL].astype(str)) |
                   set(df_4to3[GENE_COL].astype(str)) |
                   set(df_2to4[GENE_COL].astype(str)) |
                   set(df_c3[GENE_COL].astype(str)))

rows = []
for g in all_genes:
    row = {GENE_COL: g}
    for c in COLS:
        row[c] = np.nan
    for ix in [idx_meta, idx_2to3, idx_4to3, idx_c3, idx_2to4]:
        if g in ix.index:
            for c in COLS:
                if pd.isna(row[c]):
                    row[c] = float(ix.loc[g, c])
    rows.append(row)

U = pd.DataFrame(rows)

# top genes for annotation from C3 list
top_c3_annot = df_c3.sort_values("score_C3_NR_to_R", ascending=False)[GENE_COL].astype(str).head(10).tolist()

# ====== Panel A: scatter score_2to3 vs score_C3_NR_to_R ======
fig = plt.figure(figsize=(7,6))
ax = plt.gca()
ax.scatter(U["score_2to3"], U["score_C3_NR_to_R"])
ax.axhline(0, linewidth=1)
ax.axvline(0, linewidth=1)
ax.set_xlabel("score_2to3 (C3 vs C2; effector-push)")
ax.set_ylabel("score_C3_NR_to_R (R vs NR within C3; effector refinement)")
ax.set_title("Effector refinement vs effector-push")

for g in top_c3_annot:
    r = U[U[GENE_COL] == g]
    if len(r) == 1:
        r = r.iloc[0]
        ax.text(r["score_2to3"] + 0.002, r["score_C3_NR_to_R"] + 0.002, g, fontsize=8)

plt.tight_layout()
pA = os.path.join(OUTDIR, "PanelA_scatter_C3refinement_vs_2to3.png")
plt.savefig(pA, dpi=300)
plt.close(fig)

# ====== Panel B: scatter score_4to3 vs score_C3_NR_to_R ======
fig = plt.figure(figsize=(7,6))
ax = plt.gca()
ax.scatter(U["score_4to3"], U["score_C3_NR_to_R"])
ax.axhline(0, linewidth=1)
ax.axvline(0, linewidth=1)
ax.set_xlabel("score_4to3 (C3 vs C4; effector-rescue)")
ax.set_ylabel("score_C3_NR_to_R (R vs NR within C3; effector refinement)")
ax.set_title("Effector refinement vs effector-rescue")

for g in top_c3_annot:
    r = U[U[GENE_COL] == g]
    if len(r) == 1:
        r = r.iloc[0]
        ax.text(r["score_4to3"] + 0.002, r["score_C3_NR_to_R"] + 0.002, g, fontsize=8)

plt.tight_layout()
pB = os.path.join(OUTDIR, "PanelB_scatter_C3refinement_vs_4to3.png")
plt.savefig(pB, dpi=300)
plt.close(fig)

# ====== Panel C: heatmap for C3 top hits across all scores ======
hm = df_c3[[GENE_COL] + COLS].copy()
hm[GENE_COL] = hm[GENE_COL].astype(str)
hm = hm.sort_values("score_C3_NR_to_R", ascending=False).reset_index(drop=True)

mat = hm[COLS].to_numpy(dtype=float)

fig = plt.figure(figsize=(9, 0.32*len(hm) + 2))
ax = plt.gca()
im = ax.imshow(mat, aspect="auto")
ax.set_xticks(np.arange(len(COLS)))
ax.set_xticklabels(COLS, rotation=45, ha="right")
ax.set_yticks(np.arange(len(hm)))
ax.set_yticklabels(hm[GENE_COL].tolist())
ax.set_title("Top effector-refinement candidates (within C3: R vs NR)")
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("cosine similarity score")

plt.tight_layout()
pC = os.path.join(OUTDIR, "PanelC_heatmap_topC3refinement.png")
plt.savefig(pC, dpi=300)
plt.close(fig)

hm.to_csv(os.path.join(OUTDIR, "PanelC_heatmap_table_topC3refinement.csv"), index=False)

# ====== Panel D: overlap barplot (counts only) ======
set_meta = set(df_meta[GENE_COL].astype(str))
set_2to3 = set(df_2to3[GENE_COL].astype(str))
set_4to3 = set(df_4to3[GENE_COL].astype(str))
set_2to4 = set(df_2to4[GENE_COL].astype(str))
set_c3   = set(df_c3[GENE_COL].astype(str))

bars = [
    ("|C3 ∩ 2to3|", len(set_c3 & set_2to3)),
    ("|C3 ∩ 4to3|", len(set_c3 & set_4to3)),
    ("|C3 ∩ meta|", len(set_c3 & set_meta)),
    ("|meta ∩ 2to3 ∩ 4to3|", len(set_meta & set_2to3 & set_4to3)),
    ("|meta ∩ 2to4|", len(set_meta & set_2to4)),
]

fig = plt.figure(figsize=(8,4))
ax = plt.gca()
x = np.arange(len(bars))
ax.bar(x, [b[1] for b in bars])
ax.set_xticks(x)
ax.set_xticklabels([b[0] for b in bars], rotation=30, ha="right")
ax.set_ylabel("Number of shared targets")
ax.set_title("Limited overlap supports distinct regulatory axes")
plt.tight_layout()
pD = os.path.join(OUTDIR, "PanelD_overlap_bar.png")
plt.savefig(pD, dpi=300)
plt.close(fig)

print("Saved panels to:", OUTDIR)
print("A:", pA)
print("B:", pB)
print("C:", pC)
print("D:", pD)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch, Rectangle

In [ ]:
UNION_TABLE = r"/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/07_perturb_seq_T_cells/stepABC_outputs/B_heatmap_table_union_top10_plus_consensus.csv"


In [ ]:
DPI = 300

GENE_COL = "target_contrast_gene_name"
CONSENSUS4 = ["CD320", "FBXW7", "PARP8", "RANBP6"]

# score columns expected in your tables
COL_2TO3 = "score_2to3"
COL_4TO3 = "score_4to3"
COL_2TO4 = "score_2to4"
COL_C3   = "score_C3_NR_to_R"
COL_META = "meta_score"

# =========================
# Load
# =========================
df_meta = pd.read_csv(TOP_META)
df_2to3 = pd.read_csv(TOP_2TO3)
df_4to3 = pd.read_csv(TOP_4TO3)
df_2to4 = pd.read_csv(TOP_2TO4)
df_c3   = pd.read_csv(TOP_C3)
df_union = pd.read_csv(UNION_TABLE)

for df in [df_meta, df_2to3, df_4to3, df_2to4, df_c3, df_union]:
    df[GENE_COL] = df[GENE_COL].astype(str)

# =========================
# Helper: get a unified lookup for any gene from union table
# =========================
def build_lookup(df):
    d = df.drop_duplicates(GENE_COL).set_index(GENE_COL)
    return d

L = build_lookup(df_union)

def get_point(gene):
    if gene in L.index:
        r = L.loc[gene]
        return float(r[COL_2TO3]), float(r[COL_4TO3]), float(r[COL_2TO4]), float(r[COL_C3]), float(r[COL_META])
    return None

# =========================
# Panel A: Concept schematic
# =========================
def panel_A_schematic(save_path):
    fig = plt.figure(figsize=(7.2, 5.2))
    ax = plt.gca()
    ax.set_axis_off()
    ax.set_title("Panel A | Two therapeutic strategies in CD8⁺ T cells", loc="left", fontweight="bold")

    # Positions
    pos = {
        "C2\n(transitional)": (0.18, 0.55),
        "C4\n(exhausted)": (0.78, 0.30),
        "C3\n(effector)": (0.78, 0.72),
    }
    # Draw nodes
    for label, (x, y) in pos.items():
        ax.add_patch(Circle((x, y), 0.085, fill=False, linewidth=2))
        ax.text(x, y, label, ha="center", va="center", fontsize=10, fontweight="bold")

    # Arrows for strategy 1
    # C2 -> C3 (push)
    ax.add_patch(FancyArrowPatch((0.26, 0.55), (0.70, 0.72),
                                 arrowstyle="->", mutation_scale=14, linewidth=2))
    ax.text(0.48, 0.67, "push\n(C3 vs C2)", ha="center", va="center", fontsize=9)

    # C4 -> C3 (rescue)
    ax.add_patch(FancyArrowPatch((0.78, 0.39), (0.78, 0.63),
                                 arrowstyle="->", mutation_scale=14, linewidth=2))
    ax.text(0.86, 0.51, "rescue\n(C3 vs C4)", ha="left", va="center", fontsize=9)

    # C2 -> C4 (pro-exhaustion risk)
    ax.add_patch(FancyArrowPatch((0.26, 0.52), (0.70, 0.33),
                                 arrowstyle="->", mutation_scale=14, linewidth=2))
    ax.text(0.48, 0.40, "pro-exhaustion risk\n(C4 vs C2)", ha="center", va="center", fontsize=9)

    # Strategy 2: within effector NR->R
    rect = Rectangle((0.60, 0.77), 0.34, 0.18, fill=False, linewidth=1.6)
    ax.add_patch(rect)
    ax.text(0.77, 0.92, "within C3", ha="center", va="center", fontsize=9)
    ax.text(0.66, 0.84, "NR", fontsize=10, fontweight="bold")
    ax.text(0.88, 0.84, "R", fontsize=10, fontweight="bold")
    ax.add_patch(FancyArrowPatch((0.71, 0.84), (0.83, 0.84),
                                 arrowstyle="->", mutation_scale=14, linewidth=2))
    ax.text(0.77, 0.79, "effector refinement\n(R vs NR within C3)", ha="center", va="center", fontsize=8.5)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(save_path, dpi=DPI)
    plt.close(fig)

panel_A_schematic(os.path.join(OUTDIR, "panel_A_schematic.png"))

# =========================
# Panel B: Scatter (2->3 vs 4->3), meta vs pro-exhaustion
# =========================
def panel_B_scatter_push_rescue(save_path):
    fig = plt.figure(figsize=(6.4, 5.2))
    ax = plt.gca()
    ax.set_title("Panel B | Effector-push vs effector-rescue", loc="left", fontweight="bold")

    # Use union table as background
    U = df_union.drop_duplicates(GENE_COL).copy()
    meta_set = set(df_meta[GENE_COL])
    pro_set  = set(df_2to4[GENE_COL])

    m = U[U[GENE_COL].isin(meta_set)]
    p = U[U[GENE_COL].isin(pro_set)]

    ax.scatter(m[COL_2TO3], m[COL_4TO3], label="meta_top")
    ax.scatter(p[COL_2TO3], p[COL_4TO3], label="pro_exhaustion_top")

    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)

    ax.set_xlabel("score_2to3 (C3 vs C2)")
    ax.set_ylabel("score_4to3 (C3 vs C4)")
    ax.legend(frameon=True)

    # annotate consensus4
    for g in CONSENSUS4:
        if g in m[GENE_COL].values:
            rr = m[m[GENE_COL] == g].iloc[0]
            ax.text(float(rr[COL_2TO3]) + 0.003, float(rr[COL_4TO3]) + 0.003,
                    g, fontsize=10, fontweight="bold")

    # annotate a few strongest pro-exhaustion (top5 by score_2to4)
    top_p = df_2to4.sort_values(COL_2TO4, ascending=False).head(5)[GENE_COL].tolist()
    for g in top_p:
        if g in p[GENE_COL].values:
            rr = p[p[GENE_COL] == g].iloc[0]
            ax.text(float(rr[COL_2TO3]) + 0.003, float(rr[COL_4TO3]) + 0.003,
                    g, fontsize=9)

    plt.tight_layout()
    plt.savefig(save_path, dpi=DPI)
    plt.close(fig)

panel_B_scatter_push_rescue(os.path.join(OUTDIR, "panel_B_scatter_push_vs_rescue.png"))

# =========================
# Panel C: Heatmap for Strategy 1 (push, rescue, pro-exhaustion), block-ordered
# =========================
def panel_C_heatmap_strategy1(save_path):
    fig = plt.figure(figsize=(7.8, 7.2))
    ax = plt.gca()
    ax.set_title("Panel C | Score patterns for restoration vs pro-exhaustion drivers", loc="left", fontweight="bold")

    # Build a block-ordered table:
    # 1) consensus4
    # 2) meta top hits (excluding consensus)
    # 3) pro-exhaustion top hits (excluding any already used)
    meta_genes = df_meta[GENE_COL].tolist()
    pro_genes  = df_2to4[GENE_COL].tolist()

    ordered = []
    ordered += [g for g in CONSENSUS4 if g in L.index]
    ordered += [g for g in meta_genes if (g in L.index) and (g not in ordered)]
    ordered += [g for g in pro_genes if (g in L.index) and (g not in ordered)]

    # keep size manageable
    # consensus4 + meta top30 + pro top15 (typical)
    # if ordered is too long, truncate pro block
    max_rows = 4 + 30 + 15
    ordered = ordered[:max_rows]

    cols = [COL_2TO3, COL_4TO3, COL_2TO4]
    mat = L.loc[ordered, cols].to_numpy(dtype=float)

    im = ax.imshow(mat, aspect="auto")

    ax.set_xticks(np.arange(len(cols)))
    ax.set_xticklabels(["2→3 (push)", "4→3 (rescue)", "2→4 (pro-exh)"], rotation=45, ha="right")

    ax.set_yticks(np.arange(len(ordered)))
    ax.set_yticklabels(ordered, fontsize=8)

    # bold consensus
    for tick in ax.get_yticklabels():
        if tick.get_text() in CONSENSUS4:
            tick.set_fontweight("bold")

    # draw horizontal separators between blocks
    # consensus block end
    y_cons_end = len([g for g in CONSENSUS4 if g in ordered])
    y_meta_end = len([g for g in meta_genes if g in ordered]) + y_cons_end
    if y_cons_end > 0:
        ax.axhline(y_cons_end - 0.5, linewidth=1.5)
    if y_meta_end > y_cons_end:
        ax.axhline(y_meta_end - 0.5, linewidth=1.5)

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("cosine similarity")

    plt.tight_layout()
    plt.savefig(save_path, dpi=DPI)
    plt.close(fig)

panel_C_heatmap_strategy1(os.path.join(OUTDIR, "panel_C_heatmap_strategy1.png"))

# =========================
# Panel D: Strategy 2 scatter (within C3 NR->R vs rescue 4->3)
# =========================
def panel_D_scatter_refinement(save_path):
    fig = plt.figure(figsize=(6.4, 5.2))
    ax = plt.gca()
    ax.set_title("Panel D | Effector refinement (within C3 NR→R) vs exhaustion rescue", loc="left", fontweight="bold")

    U = df_union.drop_duplicates(GENE_COL).copy()

    ax.scatter(U[COL_4TO3], U[COL_C3])

    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)

    ax.set_xlabel("score_4to3 (C3 vs C4)")
    ax.set_ylabel("score_C3_NR_to_R (R vs NR within C3)")

    # annotate top10 refinement genes
    top_ref = df_c3.sort_values(COL_C3, ascending=False).head(10)[GENE_COL].tolist()
    for g in top_ref:
        if g in L.index:
            x = float(L.loc[g, COL_4TO3])
            y = float(L.loc[g, COL_C3])
            ax.text(x + 0.003, y + 0.003, g, fontsize=8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=DPI)
    plt.close(fig)

panel_D_scatter_refinement(os.path.join(OUTDIR, "panel_D_scatter_C3refinement_vs_rescue.png"))

# =========================
# Panel E: Overlap summary bars (top30 lists)
# =========================
def panel_E_overlap_summary(save_path):
    fig = plt.figure(figsize=(7.8, 4.2))
    ax = plt.gca()
    ax.set_title("Panel E | Limited overlap supports two therapeutic levers", loc="left", fontweight="bold")

    set_meta = set(df_meta[GENE_COL])
    set_2to3 = set(df_2to3[GENE_COL])
    set_4to3 = set(df_4to3[GENE_COL])
    set_2to4 = set(df_2to4[GENE_COL])
    set_c3   = set(df_c3[GENE_COL])

    bars = [
        ("meta∩2to3∩4to3", len(set_meta & set_2to3 & set_4to3)),
        ("meta∩2to4", len(set_meta & set_2to4)),
        ("C3∩meta", len(set_c3 & set_meta)),
        ("C3∩2to3", len(set_c3 & set_2to3)),
        ("C3∩4to3", len(set_c3 & set_4to3)),
    ]

    x = np.arange(len(bars))
    ax.bar(x, [v for _, v in bars])
    ax.set_xticks(x)
    ax.set_xticklabels([k for k, _ in bars], rotation=25, ha="right")
    ax.set_ylabel("shared targets (top30)")

    ymax = max(v for _, v in bars) + 2
    ax.set_ylim(0, ymax)

    for i, (_, v) in enumerate(bars):
        ax.text(i, v + 0.15, str(v), ha="center", va="bottom", fontsize=10)

    plt.tight_layout()
    plt.savefig(save_path, dpi=DPI)
    plt.close(fig)

panel_E_overlap_summary(os.path.join(OUTDIR, "panel_E_overlap_summary.png"))

print("Saved 5 panels to:", OUTDIR)
print("A:", os.path.join(OUTDIR, "panel_A_schematic.png"))
print("B:", os.path.join(OUTDIR, "panel_B_scatter_push_vs_rescue.png"))
print("C:", os.path.join(OUTDIR, "panel_C_heatmap_strategy1.png"))
print("D:", os.path.join(OUTDIR, "panel_D_scatter_C3refinement_vs_rescue.png"))
print("E:", os.path.join(OUTDIR, "panel_E_overlap_summary.png"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from adjustText import adjust_text

def panel_B_matplotlib_no_overlap(
    df_meta, df_pro, out_png,
    *,
    xcol="score_2to3",
    ycol="score_4to3",
    gene_col="target_contrast_gene_name",
    label_genes=None,          # list[str] to label; if None, label only top extremes
    label_topk=8,              # if label_genes is None: label topk by |x|+|y|
    rename_legend=True,
    dpi=300,
):
    """
    df_meta: prioritized_drivers table (your top list)
    df_pro : pro-exhaustion drivers table (your top list)
    """

    fig = plt.figure(figsize=(10, 7))
    ax = plt.gca()

    # --- scatter (keep your old style)
    ax.scatter(df_meta[xcol], df_meta[ycol], label=("prioritized_drivers" if rename_legend else "meta_top"))
    ax.scatter(df_pro[xcol],  df_pro[ycol],  label=("pro_exhaustion_top"))

    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)

    ax.set_title("Effector-push vs effector-rescue scores")
    ax.set_xlabel("score_2to3 (transitional→effector)")
    ax.set_ylabel("score_4to3 (exhausted→effector)")

    ax.legend(frameon=True)

    # --- choose which genes to label
    def _pick_default_labels(df, topk):
        tmp = df.copy()
        tmp["_rank"] = tmp[xcol].abs() + tmp[ycol].abs()
        return tmp.sort_values("_rank", ascending=False).head(topk)[gene_col].astype(str).tolist()

    if label_genes is None:
        # label topk from each group
        genes = []
        genes += _pick_default_labels(df_meta, max(3, label_topk // 2))
        genes += _pick_default_labels(df_pro,  max(3, label_topk // 2))
        # de-dup keep order
        seen = set()
        label_genes = [g for g in genes if not (g in seen or seen.add(g))]

    # build gene->(x,y) lookup from both tables
    all_df = (
        pd.concat([df_meta[[gene_col, xcol, ycol]], df_pro[[gene_col, xcol, ycol]]], axis=0)
          .drop_duplicates(gene_col)
          .set_index(gene_col)
    )

    texts = []
    for g in label_genes:
        if g not in all_df.index:
            continue
        x = float(all_df.loc[g, xcol])
        y = float(all_df.loc[g, ycol])
        texts.append(ax.text(x, y, g, fontsize=11, fontweight="bold" if g in {"CD320","FBXW7","PARP8","RANBP6"} else None))

    # --- repel labels so they don't overlap
    adjust_text(
        texts,
        ax=ax,
        # you can turn arrows on/off; arrows make it more readable when labels move far
        arrowprops=dict(arrowstyle="-", linewidth=0.8),
        expand_text=(1.12, 1.25),
        expand_points=(1.12, 1.25),
        force_text=(0.2, 0.6),
        force_points=(0.1, 0.3),
        lim=500,
    )

    plt.tight_layout()
    plt.savefig(out_png, dpi=dpi)
    plt.close(fig)


In [ ]:
import pandas as pd
# df_meta = 你的 prioritized drivers（原 meta_top 那张）
# df_pro  = 你的 pro_exhaustion_top 那张
panel_B_matplotlib_no_overlap(
    df_meta=df_meta,
    df_pro=df_pro,
    out_png=os.path.join(OUTDIR, "panel_B_matplotlib_no_overlap.png"),
    xcol="score_2to3",
    ycol="score_4to3",
    gene_col="target_contrast_gene_name",
    # 推荐：手动指定你想标注的基因（更可控）
    label_genes=["CD320","FBXW7","PARP8","RANBP6","FOXO1","UBE2L3","ZNF106","PRKAR1A","SIK3","SGF29"],
    rename_legend=True,
    dpi=300,
)